In [1]:
from pathlib import Path
import subprocess
import os
import litellm

from dotenv import load_dotenv
load_dotenv()



True

In [2]:
path = "a.txt"
context = "hahaha"

p = Path(path)
print(p)
p.parent.mkdir(parents= True, exist_ok=True)
p.write_text(context)

a.txt


6

In [3]:
r = p.read_text(encoding="utf-8")
r

'hahaha'

In [ ]:
command ="python -m hello"
result = subprocess.run(
    command,
    shell=True,
    text=True,
    capture_output=True,
    timeout=30
)
result = (result.stdout or "") + (result.stderr or "")
print(result)

Hello CLI



: 

In [ ]:
MODEL = "openai/kCode"
BASE_URL = "https://ai-gateway.inter-k.com/v1"
API_KEY = os.environ["API_KEY"]

SYSTEM_PROMPT = (
    "You are a precise coding agent."
    "Use tools when needed. Keep changes minimal and explain clearly."
)

In [ ]:

tools = [
    {
        "type": "function",
        "function": {
            "name": "read",
            "description": "Read a UTF-8 text file",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write",
            "description": "Write to a UTF-8 text file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "content": {"type": "string"}
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "edit",
            "description": "Replace exact text in a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                    "old_text": {"type": "string"},
                    "new_text": {"type": "string"}
                },
                "required": ["path", "old_text", "new_text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "bash",
            "description": "Run a shell command in the current directory",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
            },
        },
    },
]

def tool_read(path: str) -> str:
    return Path(path).read_text(encoding="utf-8")


def tool_write(path: str, content: str) -> str:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return f"wrote {path}"


def tool_edit(path: str, old_text: str, new_text: str) -> str:
    p = Path(path)
    content = p.read_text(encoding="utf-8")
    if old_text not in content:
        return f"'{old_text}' not found in {path}"
    new_content = content.replace(old_text, new_text)
    p.write_text(new_content, encoding="utf-8")
    return f"replaced '{old_text}' with '{new_text}' in {path}"


def tool_bash(command: str) -> str:
    result = subprocess.run(
        command,
        shell=True,
        text=True,
        capture_output=True,
        timeout=30
    )
    output = (result.stdout or "") + (result.stderr or "")
    return output[:8000] or "(no output)"


def execute_tool(name: str, args: dict) -> str:
    try:
        if name == "read":
            return tool_read(args["path"])
        if name == "write":
            return tool_write(args["path"], args["content"])
        if name == "edit":
            return tool_edit(args["path"], args["old_text"], args["new_text"])
        if name == "bash":
            return tool_bash(args["command"])
        return f"unknown tool: {name}"
    except Exception as e:
        return f"tool error: {e}"

In [ ]:
prompt = "hello"